# Preprocess

In [8]:
import torch
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import xarray as xr
import matplotlib.pyplot as plt

## Load BedMachine nc file

**BedMachineAntarctica-v3.nc** is 0.8 GB so we only use slices.

We ran the below (commented-out) snippet once to slice up the full data set.

45 * 500m = 22,500 km  
[50 * 450m = 22,500 km]

However, midpoints are aligned but edges are not. Edges never line up as the ice velocity edges run through 25ers and the BedMachine edges run through 50er.

In [11]:
"""
### Load original dataset ###
# Load from whereever it is stored
bed = xr.open_dataset("/Users/kimbente/BedMachineAntarctica-v3.nc")

### DOMAINS ###
scene_size = 22500 # in meters
n_scenes = 30 # scenes per row/column: 900 scences per domain
span = scene_size * n_scenes

# Transantarctic mountains: Nimrod, Byrd, Skelton glacier, Victoria land
# mountainous domain spanning grounded ice and floating ice.
# stick to order: y, x
transant_y_min = - 1232000
transant_x_min = 3500 # Changed
transant_y_max = transant_y_min + span # -557000
transant_x_max = transant_x_min + span # 678500

# Dome C
# lake Vostok is still further north than this domain
domec_y_min = -1232000
domec_x_min = 899000
domec_y_max = domec_y_min + span # -557000
domec_x_max = domec_x_min + span # 1574000

### Create slices ###
# reduces data size down to managable level
# y slicing ordering is (max, min)
bed_transant_slice = bed.sel(x = slice(transant_x_min, transant_x_max), y = slice(transant_y_max, transant_y_min))
bed_domec_slice = bed.sel(x = slice(domec_x_min, domec_x_max), y = slice(domec_y_max, domec_y_min))

### Save ###

bed_transant_slice.to_netcdf(path = './nc_data/BedMachineAntarctica-v3_TransantarcticMountains_slice.nc')
bed_domec_slice.to_netcdf(path = './nc_data/BedMachineAntarctica-v3_DomeC_slice.nc')
"""

In [12]:
bed_transant_slice = xr.open_dataset('./nc_data/BedMachineAntarctica-v3_TransantarcticMountains_slice.nc')
bed_domec_slice = xr.open_dataset('./nc_data/BedMachineAntarctica-v3_DomeC_slice.nc')

In [14]:
bed_domec_slice

<xarray.Dataset>
Dimensions:    (x: 1351, y: 1351)
Coordinates:
  * x          (x) int32 899000 899500 900000 900500 ... 1573000 1573500 1574000
  * y          (y) int32 -557000 -557500 -558000 ... -1231000 -1231500 -1232000
Data variables:
    mapping    |S1 ...
    mask       (y, x) int8 ...
    firn       (y, x) float32 ...
    surface    (y, x) float32 ...
    thickness  (y, x) float32 ...
    bed        (y, x) float32 ...
    errbed     (y, x) float32 ...
    source     (y, x) int8 ...
    dataid     (y, x) int8 ...
    geoid      (y, x) int16 ...
Attributes: (12/17)
    Conventions:                 CF-1.7
    Title:                       BedMachine Antarctica
    Author:                      Mathieu Morlighem
    version:                     03-Jun-2022 (v3.4)
    nx:                          13333.0
    ny:                          13333.0
    ...                          ...
    ymax:                        3333000
    spacing:                     500
    no_data:                     -9999.0
    license:                     No restrictions on access or use
    Data_citation:               Morlighem M. et al., (2019), Deep glacial tr...
    Notes:                       Data processed at the Department of Earth Sy...

# Convert to scenes

In [22]:
domain_dims = 1351

# Domain tensor [C, 1350, 1350] with C = 6
# Unsqueeze for concatination.
domec_tensor = torch.cat((torch.tensor(bed_domec_slice.bed.values).unsqueeze(0),
                          torch.tensor(bed_domec_slice.surface.values).unsqueeze(0),
                          torch.tensor(bed_domec_slice.thickness.values).unsqueeze(0),
                          torch.tensor(bed_domec_slice.mask.values).unsqueeze(0),
                          torch.tensor(bed_domec_slice.firn.values).unsqueeze(0),
                          torch.tensor(bed_domec_slice.errbed.values).unsqueeze(0),
                          # YX
                          torch.tensor(bed_domec_slice.coords["y"].values).unsqueeze(-1).repeat(1, domain_dims).unsqueeze(0),
                          torch.tensor(bed_domec_slice.coords["x"].values).repeat(domain_dims, 1).unsqueeze(0)),
                          dim = 0)

In [23]:
def domain_to_scenes(domain_tensor, scene_hw = 50):
    """Works for an arbitrary number of channels.
    Args:
        domain_tensor (_type_): _description_
        scene_hw (int, optional): _description_. Defaults to 50.

    Returns:
        _type_: _description_
    """

    n_channels = domain_tensor.shape[0]
    domain_hw = domain_tensor.shape[-1]
    n_hw = int(domain_hw / scene_hw)  

    # Initailise empty scene tensor
    # [N, C, H, W] where n can be split into batches. N (dim 0) can be zero as we concat along this axis.
    scene_tensor = torch.empty(size = (0, n_channels, scene_hw, scene_hw))

    for row in range(0, n_hw):
        row_min = row * scene_hw
        row_max = row_min + scene_hw

        for column in range(0, n_hw):
            column_min = column * scene_hw
            column_max = column_min + scene_hw

            scene_tensor = torch.cat((scene_tensor, domain_tensor[:, row_min : row_max, column_min : column_max].unsqueeze(0)), dim = 0)
    
    return scene_tensor

In [25]:
# Go from [C, 1350, 1350] to [900, C, 45, 45]: 
# 1350 * 1350 == 900 * 45 * 45 [1,822,500]
# Conversion can take ~ 11 sec.
domec_scene_tensor = domain_to_scenes(domec_tensor, scene_hw = 45)
print(domec_scene_tensor.shape)

torch.save(domec_scene_tensor, './torch_data/DOMEC_bed_scenes.pt')

torch.Size([900, 8, 45, 45])


## Repeat for Transantactic domain

In [27]:
domain_dims = 1351

# Domain tensor [C, 1350, 1350] with C = 6
# Unsqueeze for concatination.
transant_tensor = torch.cat((torch.tensor(bed_transant_slice.bed.values).unsqueeze(0),
                             torch.tensor(bed_transant_slice.surface.values).unsqueeze(0),
                             torch.tensor(bed_transant_slice.thickness.values).unsqueeze(0),
                             torch.tensor(bed_transant_slice.mask.values).unsqueeze(0),
                             torch.tensor(bed_transant_slice.firn.values).unsqueeze(0),
                             torch.tensor(bed_transant_slice.errbed.values).unsqueeze(0),
                             # YX
                             torch.tensor(bed_transant_slice.coords["y"].values).unsqueeze(-1).repeat(1, domain_dims).unsqueeze(0),
                             torch.tensor(bed_transant_slice.coords["x"].values).repeat(domain_dims, 1).unsqueeze(0)),
                             dim = 0)

In [28]:
# Go from [C, 1350, 1350] to [900, C, 45, 45]: 
# 1350 * 1350 == 900 * 45 * 45 [1,822,500]
# Conversion can take ~ 11 sec.
transant_scene_tensor = domain_to_scenes(transant_tensor, scene_hw = 45)
print(transant_scene_tensor.shape)

torch.save(transant_scene_tensor, './torch_data/TRANSANT_bed_scenes.pt')

torch.Size([900, 8, 45, 45])


# 45 Scences

In [18]:
# Select top left corner for 1 scene
scene = bed_transant_slice.isel(y = slice(0, 45), x = slice(0, 45))

# to_pandas() contains matrix format
# to_dataframe() keeps multi-index
# .values outputs array which can be converted to tensor

scene_bed_tensor = torch.tensor(scene.bed.values)
scene_sur_tensor = torch.tensor(scene.surface.values)

torch.save(scene_bed_tensor, './torch_data/scene_bed_tensor.pt')
torch.save(scene_sur_tensor, './torch_data/scene_sur_tensor.pt')

## 46er scene

surface values

In [19]:
dims = 46

scene46 = bed_transant_slice.isel(y = slice(0, dims), x = slice(0, dims))
scene46_sur_tensor = torch.tensor(scene46.surface.values)
torch.save(scene46_sur_tensor, './torch_data/scene46_sur_tensor.pt')

In [20]:
# Increase to 46 to span full area
scene_sur_midpoints46_tensor = torch.cat((torch.tensor(scene46.coords["y"].values).unsqueeze(-1).repeat(1, dims).unsqueeze(0),
                              torch.tensor(scene46.coords["x"].values).repeat(dims, 1).unsqueeze(0)),
                              dim = 0)

torch.save(scene_sur_midpoints46_tensor, './torch_data/scene_sur_midpoints46_tensor.pt')

## Visualise values

In [21]:
fig = px.imshow(scene_bed_tensor, 
                color_continuous_scale = 'RdBu_r',
                # DEFAULT for matrices: only tensor because of conversion
                origin = "upper", title = "Bed elevation of scene one")
fig.show()

In [22]:
fig = px.imshow(scene_sur_tensor, 
                color_continuous_scale = 'RdBu_r',
                # DEFAULT for matrices: only tensor because of conversion
                origin = "upper", title = "Surface of scene one")
fig.show()

In [27]:
# Magnify data because it is too large to visualise
magnification_factor = 10
magnify = torch.nn.AvgPool2d(kernel_size = magnification_factor)
# stride is my default the kernel size

input = scene_bed_tensor.unsqueeze(0)
# input = input.type(torch.DoubleTensor)
bed_magnified = magnify(input)
bed_magnified.shape

torch.Size([1, 4, 4])

In [28]:
fig = px.imshow(bed_magnified.squeeze(), 
                color_continuous_scale = 'RdBu_r',
                # DEFAULT for matrices: only tensor because of conversion
                origin = "upper", title = "Bed elevation of the Transantarctic mountains")
fig.show()

In [32]:
fig = px.imshow((scene.mask.values), 
                # DEFAULT for matrices: only tensor because of conversion
                origin = "upper", 
                title = "Landtypes: 1: ice free land, 2 grounded ice")
fig.show()

In [33]:
scene.source.values

array([[5, 3, 3, ..., 5, 5, 3],
       [5, 5, 3, ..., 5, 5, 5],
       [5, 5, 5, ..., 5, 5, 5],
       ...,
       [5, 5, 5, ..., 5, 5, 5],
       [5, 5, 5, ..., 5, 5, 5],
       [5, 5, 5, ..., 5, 5, 5]], dtype=int8)

In [34]:
fig = px.imshow((scene.firn.values), 
                # DEFAULT for matrices: only tensor because of conversion
                origin = "upper", 
                title = "Firn air content (in meters) correction value")
fig.show()

In [36]:
fig = px.imshow((scene.errbed.values), 
                # DEFAULT for matrices: only tensor because of conversion
                origin = "upper", 
                title = "Firn air content (in meters) correction value")
fig.show()

In [37]:
# Check: this should hold true
(scene.surface.values - scene.bed.values - scene.thickness.values)

fig = px.imshow((scene.surface.values - scene.bed.values - scene.thickness.values), 
                color_continuous_scale = 'RdBu_r',
                # DEFAULT for matrices: only tensor because of conversion
                origin = "upper", title = "Floating ice")
fig.show()